# Debug da pipeline de segmentação

Notebook para inspecionar cada etapa da detecção de etiquetas escuras nas embalagens.


## 1. Setup e carregamento dos dados


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pdiseg

ROOT = Path('.').resolve()
DATASET = ROOT / 'data' / 'Train_and_Validation'
RESULT = ROOT / 'result'
DEBUG_RESULT = ROOT / 'debug_result'

DATASET, RESULT


In [6]:
classes = pdiseg.list_classes(DATASET)
counts = pdiseg.count_images_per_class(DATASET)
summary = pd.DataFrame({'class_name': classes, 'images': [counts.get(c, 0) for c in classes]})
summary


NameError: name 'pdiseg' is not defined

In [ ]:
# escolha manual de classe e índice
CLASS_NAME = classes[0]
paths = [p for p in pdiseg.find_source_images(DATASET) if p.parent.name == CLASS_NAME]
IMAGE_PATH = paths[0]
IMAGE_PATH


In [ ]:
image = pdiseg.load_image(IMAGE_PATH)
plt.figure(figsize=(10, 5))
plt.imshow(pdiseg.to_rgb(image))
plt.title(IMAGE_PATH.name)
plt.axis('off')


## 2. Diagnóstico da base


In [ ]:
rows = []
for path in pdiseg.find_source_images(DATASET)[:200]:
    img = pdiseg.load_image(path)
    rows.append({
        'class': path.parent.name,
        'file': path.name,
        'h': img.shape[0],
        'w': img.shape[1],
        'mean': float(img.mean()),
        'std': float(img.std()),
        'p95': float(np.percentile(img, 95)),
    })
diag = pd.DataFrame(rows)
diag.groupby('class').agg(images=('file', 'count'), mean=('mean', 'mean'), std=('std', 'mean'), p95=('p95', 'mean')).head(10)


## 3. Visualização da pipeline atual


In [ ]:
detection, prep, masks = pdiseg.debug_frame(image)
print('candidates', len(detection.candidates), 'kept', len(detection.kept), 'labels', len(detection.labels))
pd.DataFrame(pdiseg.scored_table(detection.scored)).head(10)


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes[0,0].imshow(pdiseg.to_rgb(image)); axes[0,0].set_title('original')
axes[0,1].imshow(prep.work, cmap='gray'); axes[0,1].set_title('preprocessado (CLAHE)')
axes[0,2].imshow(pdiseg.draw_boxes(image, detection.candidates, (220,50,50))); axes[0,2].set_title('candidatos brutos')
mask_views = pdiseg.visualize_masks(masks)
axes[1,0].imshow(mask_views['combined'], cmap='gray'); axes[1,0].set_title('máscara combinada')
axes[1,1].imshow(pdiseg.draw_boxes(image, detection.kept, (240,200,40))); axes[1,1].set_title('aprovados (score)')
axes[1,2].imshow(pdiseg.draw_boxes(image, detection.labels, (40,200,80))); axes[1,2].set_title('labels finais')
for ax in axes.ravel(): ax.axis('off')
plt.tight_layout()


In [ ]:
ncols = max(1, len(detection.labels))
fig, axes = plt.subplots(1, ncols, figsize=(3*ncols, 3))
if ncols == 1:
    axes = [axes]
for i, box in enumerate(detection.labels, start=1):
    crop = pdiseg.crop(image, box)
    axes[i-1].imshow(crop, cmap='gray')
    axes[i-1].set_title(f'crop {i}')
    axes[i-1].axis('off')
plt.tight_layout()


## 4. Pré-processamento


In [ ]:
from pdiseg.preprocess import shadow_corrected
from skimage.exposure import adjust_gamma

variants = {
    'original': image,
    'clahe_work': prep.work,
    'shadow_corr': shadow_corrected(image),
    'gamma_08': np.clip(image.astype(np.float32) ** 0.8, 0, 255).astype(np.uint8),
}
fig, axes = plt.subplots(1, len(variants), figsize=(4*len(variants), 4))
for ax, (name, img) in zip(axes, variants.items()):
    ax.imshow(img, cmap='gray'); ax.set_title(name); ax.axis('off')
plt.tight_layout()


## 5. Máscaras e candidatos


In [ ]:
cfg = pdiseg.DetectionConfig()
raw_boxes = pdiseg.find_candidate_boxes(prep.work, cfg, text_source=prep.gray)
geometry = pdiseg.keep_label_clusters(raw_boxes, cfg)
len(raw_boxes), len(geometry)


## 6. Scoring dos candidatos


In [ ]:
scored = pd.DataFrame(pdiseg.scored_table(detection.scored))
scored[['rank','score','area_frac','dark_density','glare_fraction','contrast']].head(12)


## 7. Pós-processamento


## 11. Testes e validação

Checks rápidos (espelham `tests/test_detector_validation.py`):

In [ ]:
assert len(classes) >= 1
report = pdiseg.process_dataset(DATASET, RESULT)
assert report.total_frames > 0
assert report.total_crops >= 0
print('OK:', report.total_frames, 'frames,', report.total_crops, 'crops,', report.empty_frames, 'empty')

## 12. Limitações e estratégia

- **Limitações:** reflexos fortes, plástico muito amassado, impressões escuras que não são o nome do produto, e etiquetas parcialmente ocluídas ainda podem gerar falso positivo ou frame vazio.
- **Estratégia:** priorizar poucos crops com score alto (`max_labels_per_frame=2`), text-density no gray original, e fallback só com score mínimo — preferimos omitir crop ruim a inventar detecção.
- **Próximo passo humano:** usar `make review` para validar visualmente classes difíceis (Sassami, selados, reflexivos).

In [ ]:
labels, scored_all, kept = pdiseg.postprocess_boxes(image, prep.work, raw_boxes, cfg)
print('kept', len(kept), 'labels', len(labels))


## 8. Debug visual por classe


In [ ]:
sample_paths = [p for p in pdiseg.find_source_images(DATASET) if p.parent.name == CLASS_NAME][:6]
fig, axes = plt.subplots(len(sample_paths), 2, figsize=(10, 3*len(sample_paths)))
for row, path in enumerate(sample_paths):
    img = pdiseg.load_image(path)
    det = pdiseg.inspect_detection(img)
    axes[row,0].imshow(pdiseg.to_rgb(img)); axes[row,0].set_title(path.name)
    axes[row,1].imshow(pdiseg.draw_boxes(img, det.labels, (40,200,80)))
    axes[row,1].set_title(f'{len(det.labels)} labels')
    axes[row,0].axis('off'); axes[row,1].axis('off')
plt.tight_layout()


## 9. Salvamento


In [ ]:
# salvar crops oficiais
# pdiseg.run(DATASET, RESULT)

# salvar debug de um frame
pdiseg.save_debug_bundle(image, DEBUG_RESULT / CLASS_NAME, IMAGE_PATH.stem)


## 10. Relatório da base


In [ ]:
report = pdiseg.process_dataset(DATASET, RESULT)
rows = [{
    'class': r.class_name,
    'frames': r.frames,
    'crops': r.crops,
    'empty': r.empty_frames,
    'avg': r.crops / max(r.frames, 1),
} for r in report.classes]
base_report = pd.DataFrame(rows).sort_values('empty', ascending=False)
print('total frames', report.total_frames, 'crops', report.total_crops, 'empty', report.empty_frames)
base_report
